In [1]:
import os
import sys
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf spark.driver.extraClassPath="C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar" '
    "pyspark-shell"
)

# 2. Ensure Python paths align for the worker processes
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

# --- Define Paths ---
# Adjust the local path to match your OS requirements (e.g., C:/data/... for Windows)
RPT_WAREHOUSE_PATH_LOCAL = "/data/data_files/iceberg/WideWorldImportersDW"
RPT_WAREHOUSE_PATH_MINIO = "s3a://iceberg/iceberg/WideWorldImportersDW"

# --- 3. Initialize Combined SparkSession ---
spark = SparkSession.builder \
    .appName("Iceberg Local to MinIO Transfer") \
    .config("spark.jars.packages", 
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    \
    .config("spark.sql.catalog.local_rpt", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local_rpt.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config("spark.sql.catalog.local_rpt.warehouse", f"file:///{RPT_WAREHOUSE_PATH_LOCAL.lstrip('/')}") \
    \
    .config("spark.sql.catalog.minio_rpt", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.minio_rpt.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config("spark.sql.catalog.minio_rpt.warehouse", RPT_WAREHOUSE_PATH_MINIO) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

In [ ]:



# MinIO needs the namespaces to exist before creating tables inside them
spark.sql("CREATE NAMESPACE IF NOT EXISTS minio_rpt.integration")



In [ ]:
tables_to_copy = {
    "dimension": [
        "payment_method", "supplier", "city", 
        "stock_item", "customer", "transaction_type", "employee"
    ],
    "fact": [
        "purchase", "stock_holding", "order", 
        "movement", "sale", "transaction"
    ]
}